<table style="width:100%">
  <tr>
    <td valign="top"><img src="../data/img/FER_logo_2.png" width=300 height=80 align="left"></td>
    <td valign="top"><img src="../data/img/LARES_2_transparent.png" width=250 height=80 align="right"></td>
  </tr>
 </table>

# Exploratory data analysis (EDA) and data processing

## Part 0 - Setup

Run this cell first, in every notebook. It fetches the course repository into
the Colab session and moves into the `notebooks/` folder, so that the
`../data/...` paths work.

It is safe to run more than once, and safe after a restart.

Note: outside Colab the cell does nothing except report the working directory.
Start Jupyter from inside `notebooks/` and the paths work the same way.

If it prints `data ok: True`, we are set.

In [ ]:
# --- SETUP: run this first ---
# works in Colab and locally, safe to run more than once
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/unizg-fer-lares/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))

## Import libraries
Import all the necessary Python libraries.
Set some general options for plotting, suppress warnings etc.

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import norm

In [ ]:
import seaborn as sns
sns.set_theme(style="whitegrid")
import matplotlib.pyplot as plt
tex_fonts = {
    "font.family": "serif",
    # Use 26pt font in plots
    "axes.labelsize": 20,
    "font.size": 20,
    "figure.titlesize": 20,
    # Make the legend/label fonts a little smaller
    "legend.title_fontsize": 18,
    "legend.fontsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18
}

plt.rcParams.update(tex_fonts)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Ames Housing dataset

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this playground dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence.

With 79 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, our **goal is to predict the final price of each home**.

The Ames Housing dataset was compiled by Dean De Cock for use in data science education. It's an incredible alternative for data scientists looking for a modernized and expanded version of the often cited Boston Housing dataset. The dataset, along many exemplary notebooks can be found on [Kaggle](https://www.kaggle.com/c/house-prices-advanced-regression-techniques). Keep in mind, the Kaggle version of the dataset keeps 50% of house prices hidden for scoring purposes - full version of the dataset can be found [here](https://www.kaggle.com/datasets/prevek18/ames-housing-dataset).

![title](../data/img/house_prices.jpg)

## Load dataset

In [ ]:
# load the stored dataset from the data/ folder and print out few of the first rows for easier visual checking.
df = pd.read_csv('../data/housing_prices/housing.csv',index_col='Id')
df.head()

In [ ]:
# dataframe properties
df.info()

## Part 1 - Check basic dataset properties
Check some basic properties of the dataset to get a better sense of the data we are dealing with such as:
- check the size of the dataset (rows - data recordings, columns - inputs/features)
- check what are the available inputs/features (column names)
- check the types of available inputs/features
- check some basic statistical properties of the data
- missing values

In [ ]:
# dataset size (rows, columns)
print('Size of dataset', df.shape)

In [ ]:
# dataset columns (possible features)
print(len(df.columns))
print(df.columns.values)

In [ ]:
# dataset column types
df.dtypes.value_counts()

In [ ]:
# basic dataset statistics
df.describe()

In [ ]:
# make it a little bit easier on the eyes :)
df.describe().transpose()

**Conclusions**:
- *lot of possible features (80), 37 numeric, 43 categorical: doesn't seem likely that we will use them all,*
- *different ranges within the feature set (e.g. LotArea [1300, 215245], OverallQual [1, 10]),*
- *missing values for some features.*

### Missing values
Check and handle (remove or fill - if possible) missing values of data.

In [ ]:
# find the columns with the largest amounts of missing data
total = df.isnull().sum().sort_values(ascending = False)
percent = (df.isnull().sum()/df.isnull().count()*100).sort_values(ascending = False)
missing_data  = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_data.head(20)

In [ ]:
# remove all features with over 0.5% of missing data
df = df.drop((missing_data[missing_data['Percent'] > 0.5]).index, axis=1)
# remove the single data with missing 'Electrical' data
df = df.drop(df.loc[df['Electrical'].isnull()].index)
# check the new dataset size
print('Size of dataset', df.shape)

**Conclusions**:
- *large number of sparse (unusablle) features,*
- *removed 18 features and one row with missing Electrical.*

> # HANDS-ON: Missing values handling
>
> We just dropped **18 features** because more than 0.5% of their values were
> missing. That is a threshold we chose, not a general rule.
>
> - Would you have used a different threshold? What does 5% cost you compared
>   with 0.5%?
> - `LotFrontage` was missing 17.7% and is now gone. `GarageQual` was missing
>   5.5%. Would you have imputed either of them instead?
> - What is the argument for keeping a column that is 95% empty?
>
> Two sentences in the cell below. No code needed.

### Your answer

<!-- write two or three sentences here -->

**Answer:**


> # HANDS-ON: Missing values?
> `FireplaceQu` was dropped because 47% of its values are missing. Check that
> number against a second column before accepting it:
>
> ```python
> (df['FireplaceQu'].isna() == (df['Fireplaces'] == 0)).mean()
> ```
>
> - What does a missing value in `FireplaceQu` actually mean?
> - The same pattern holds for `PoolQC`, `GarageQual` and `BsmtQual`. What do
>   those four columns have in common?
> - Houses with a fireplace have a median price of 191 000 $, those without
>   135 000 $. What did dropping the column cost?
> - How would you handle it instead? One line is enough.
>
> Note: "percentage missing" is a property of the file, not of the houses.

### Your answer

<!-- write two or three sentences here -->

**Answer:**


## Part 2 - Exploratory data analysis (EDA)
Take a closer look into the available dataset:
- analyze the variable we are trying to predict in more detail (basic statistics, distribution)
- correlations between variables
- outliers?
- closer look into the most likely features

### Sale Price variable

In [ ]:
# basic statistics (again)
df['SalePrice'].describe()

In [ ]:
# scatter plot for all sales price data
plt.figure(figsize=(25,6))
plt.scatter(df.index,df['SalePrice'], s=50)
plt.xlim(0,1500)
plt.xlabel('Id')
plt.ylabel('SalePrice');

In [ ]:
#histogram and normal probability plot
plt.figure(figsize=(20,6))
sns.histplot(df['SalePrice'], kde=True);

In [ ]:
#skewness and kurtosis
print("Skewness: %f" % df['SalePrice'].skew()) 
print("Kurtosis: %f" % df['SalePrice'].kurt())

The histogram is an effective graphical technique for showing both the skewness and kurtosis of data set.

Skewness is a measure of symmetry, or more precisely, the lack of symmetry. The skewness for a normal distribution is zero, and any symmetric data should have a skewness near zero (negative - skewed left, positive - skewed right).

Kurtosis is a measure of whether the data are heavy-tailed or light-tailed relative to a normal distribution. The kurtosis for a standard normal distribution is three. Pandas uses Fisher’s definition of kurtosis (kurt-3).

**Conclusions**:
- *skewed distribution of the target variable*,
- *outliers present*,
- *some houses with very high prices.*

### Correlations
Correlation coefficients are used to measure how strong a relationship is between two variables. There are several types of correlation coefficient, but the most popular is Pearson’s.

In [ ]:
# calculate and plot all correlations...
corr = df.corr(numeric_only=True)
plt.figure(figsize=(12,10))
sns.heatmap(corr, xticklabels=corr.columns, yticklabels=corr.columns);

In [ ]:
# sort the correlations between features and Sale price
corr['SalePrice'].sort_values(ascending=False)

In [ ]:
# plot correlations of the 10 most correlated features
relevant_inp = np.abs(corr['SalePrice']).sort_values(ascending=False).index[0:11]
relevant_corr = df[relevant_inp].corr(numeric_only=True)

# create a mask for the lower corr triangle only
mask = np.zeros_like(relevant_corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

# want diagonal elements as well
mask[np.diag_indices_from(mask)] = False

# plot the corr heatmap
plt.figure(figsize=(9,7))
sns.heatmap(relevant_corr, xticklabels=relevant_corr.columns, yticklabels=relevant_corr.columns,
        mask=mask, square=True, annot=True, linewidths=.5, cbar_kws={"shrink": .8}, annot_kws={"fontsize":14});

**Conclusions**:
- *a lot of uncorrelated variables that can be omitted,*
- *high correlation with the expected variables (common knowledge): overall house quality, living area, garage, bathroom, year built and remodelled...*
- *some variable highly correlate among themselves and perhaps can be merged or reduced (garage area and no. of garage cars).*

> # HANDS-ON: Feature selection
>
> We ranked features by correlation with `SalePrice` and kept the top ten. A reasonable default with a large blind spot:
>
> - `df.corr()` was called with `numeric_only=True`. This dataset has 43
>   **categorical** columns. Where are they in this ranking?
> - `GarageArea` and `GarageCars` correlate strongly with each other. Is
>   keeping both an advantage or a problem?
> - Suppose a column recorded the estate agent's own valuation. It would top
>   this ranking. Would you keep it?

### Your answer

<!-- write two or three sentences here -->

**Answer:**


### Outliers
In statistics, an outlier is a data point that differs significantly from other observations.

In [ ]:
# remember calculated correlations
relevant_inp[1:]

In [ ]:
# plot pairwise relationships in a dataset
g = sns.pairplot(df[['SalePrice','OverallQual','GrLivArea','GarageArea','YearBuilt']])
g.fig.set_size_inches((15,15))

In [ ]:
# plot box plots of sales prices with respect to the house overall quality ratings
plt.figure(figsize=(10,6))
g = sns.boxplot(x=df['OverallQual'],y=df['SalePrice'])

A box plot (or box-and-whisker plot) shows the distribution of quantitative data in a way that facilitates comparisons between variables or across levels of a categorical variable. The box shows the quartiles of the dataset while the whiskers extend to show the rest of the distribution, except for points that are determined to be “outliers” using a method that is a function of the inter-quartile range.

In [ ]:
# box plots of sales prices with respect to house built year
plt.figure(figsize=(25,8))
g = sns.boxplot(x=df['YearBuilt'],y=df['SalePrice'], order=np.sort(df['YearBuilt'].unique()))
plt.xticks(rotation=45, ha='right');

**Conclusions**:
- *outliers present in most of the highly correlated features,*
- *since visual analysis shows good results, remove outliers manualy (e.g., two houses with almost largest living areas and bellow average prices),*
- *further investigate suspicious data and try to find out if there are some other reasons that are not shown in bivariate analysis (e.g., were the oldest houses with high prices been recently remodelled, or have large living areas, etc.).*

### Conclusions - overall:
- *lot of possible features (79), 36 numeric, 43 categorical: doesn't seem likely that will use them all,*
- *different ranges within the feature set (e.g. LotArea [1300, 215245], OverallQual [1, 10]),*
- *missing values for some features,*
- *large number of sparse (unusablle) features,*
- *removed 18 features and one row with missing Electrical,*
- *skewed distribution of the target variable*,
- *outliers present*,
- *some houses with very high prices,*
- *a lot of uncorrelated variables that can be omitted,*
- *high correlation with the expected variables (common knowledge): overall house quality, living area, garage, bathroom, year built and remodelled...*
- *some variable highly correlate among themselves and perhaps can be merged or reduced (garage area and no. of garage cars),*
- *outliers present in most of the highly correlated features,*
- *since visual analysis shows good results, remove outliers manualy (e.g., two houses with almost largest living areas and bellow average prices),*
- *further investigate suspicious data and try to find out if there are some other reasons that are not shown in bivariate analysis (e.g., were the oldest houses with high prices been recently remodelled, or have large living areas, etc.).*

## Part 3 - Process the training dataset
Once the available dataset is analysed with more detail, transform it into the appropriate form suitable for learning various models:
- categorical data encoding
- scaling/transforming
- splitting into train/val/test datasets

### Categorical data

In [ ]:
# scikit-learn categorical variables encoders
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [ ]:
# check if there are any categorical features in the most correlated ones
df[relevant_inp[1:]].head()

In [ ]:
# find other categorical features
categorical_var = df.select_dtypes(include='object')
categorical_var.columns

In [ ]:
# let's see the 'Heating' categories
df['Heating'].unique()

In [ ]:
# let's see the 'MSZoning' categories
df['MSZoning'].unique()

#### Label encoder

In [ ]:
# use scikit-learn label encoder and create an instance for the 'Heating' categories
le = LabelEncoder()
aux = df['Heating']
aux = le.fit_transform(aux)
np.unique(aux)

In [ ]:
# print the value-category map
le.classes_

#### One-hot encoder

In [ ]:
# use scikit-learn one-hot encoder and create an instance for both 'Heating' and 'MSZoning' categories
enc = OneHotEncoder(handle_unknown='ignore')
aux = df[['Heating']]
df_aux = enc.fit_transform(aux)
df_aux

> # HANDS-ON: Ordinal categories
>
> `LabelEncoder` on `Heating` is harmless - the categories have no order, and
> the model treats the numbers as labels.
>
> Now apply the same encoder to `ExterQual`, which does have an order, and look
> at what you get:
>
> ```python
> le.fit(df['ExterQual'].dropna())
> print(list(le.classes_))
> print(df.groupby('ExterQual')['SalePrice'].median().sort_values())
> ```
>
> - The encoder assigns numbers in alphabetical order. What order does the
>   price suggest?
> - A linear model reads those numbers as quantities. What has it learned about
>   exterior quality?
> - `Heating` has 6 categories, `Neighborhood` has 25. Does one-hot encoding
>   have a cost that grows with either of those numbers?
>
> Note: the choice is not between `LabelEncoder` and `OneHotEncoder`. It is
> between nominal, ordinal and high-cardinality columns, and each wants
> something different.

<!-- your answer here -->

**Answer:**


### Dataset splitting
First we split the data into inputs (features) and outputs (targets).

Then we further split into the train-val-test parts of the dataset. The train-val-test split procedure is used to estimate the performance of machine learning algorithms when they are used to make predictions on data not used to train the model.

In [ ]:
# split into intputs and outputs
# use only the 10 most correlated features (simplification for presentation purposes)
X = df[relevant_inp[1:]]
y = df[relevant_inp[0]]

In [ ]:
# use scikit-learn dataset splitter
from sklearn.model_selection import train_test_split

# split the dataset into train - test dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=1)

# separate dataset for validation?
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, shuffle=True, random_state=1) # 0.25 x 0.8 = 0.2

print('Train dataset size:', X_train.shape[0])
print('Validation dataset size:', X_val.shape[0])
print('Test dataset size:', X_test.shape[0])

In [ ]:
# keep only the original train-test split for future use
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=1)

print('Train dataset size:', X_train.shape[0])
print('Test dataset size:', X_test.shape[0])

> # HANDS-ON: Dataset splitting
>
> `train_test_split` with `shuffle=True` divides the houses at random. Nothing
> guarantees that both sides look alike.
>
> There are 18 houses with `OverallQual == 10` in the dataset. Repeat the split
> with 200 different `random_state` values and the model scored each time:
>
> ```
> quality-10 houses landing in train : min 8, max 18  (of 18)
> the most expensive house falls in test : 41 of 200 splits
> R2 on the test set : min 0.532   median 0.784   max 0.845
> ```
>
> - The same code and the same data give R2 anywhere from 0.53 to 0.85. Which one would you report?
> - When the most expensive house is in the test set, the model has never seen a house like it. Is a low score there a model problem or a split problem?
> - The split can also be uneven in quality, neighbourhood, or year of construction, not only in price. Which of those would worry you most here?
> - What would you change about the split, and what does `stratify` do for a continuous target like price?

<!-- your answer here -->

**Answer:**


### Dataset scaling/transforming
Some machine learning algorithms are sensitive to feature scaling while others are virtually invariant to it. 

Machine learning algorithms like linear regression, logistic regression, neural network, etc. that use gradient descent as an optimization technique require data to be scaled. Distance algorithms like KNN, K-means, and SVM are most affected by the range of features. This is because behind the scenes they are using distances between data points to determine their similarity.

Distance algorithms like KNN, K-means, and SVM are most affected by the range of features. This is because behind the scenes they are using distances between data points to determine their similarity.

**NOTE:** we will proceed with the overall dataset (train and test parts together) only for presentation purposes. Additionally, we will not save the scaled versions of the data since the scaling is model dependent as said above.

In [ ]:
# import scikit-learn scalers
from sklearn.preprocessing import MinMaxScaler, StandardScaler, QuantileTransformer, PowerTransformer

In [ ]:
# log transformation
aux = np.log(df['SalePrice'])
plt.figure(figsize=(10,6))
sns.histplot(aux, kde=True);

The log transform fixes the skew almost completely (1.88 to 0.12). We are
**not** applying it to the saved target here: whether to model `SalePrice` or
`log(SalePrice)` is a modelling choice, and we make it on day 2.

Write the observation down now so it is not lost: *the target is right-skewed;
a log transform is available and worth testing.*

In [ ]:
# scikit-learn scalers transformations and plotting
plt.figure(figsize=(8,6))
sns.histplot(MinMaxScaler().fit_transform(df['SalePrice'].values.reshape(-1, 1)),label='MinMax')
sns.histplot(StandardScaler().fit_transform(df['SalePrice'].values.reshape(-1, 1)),label='Standard')
sns.histplot(QuantileTransformer(output_distribution='uniform').fit_transform(df['SalePrice'].values.reshape(-1, 1)),label='Quantile')
sns.histplot(PowerTransformer(method='yeo-johnson').fit_transform(df['SalePrice'].values.reshape(-1, 1)),label='Yeo-Johnson')
plt.xlim([-3,3])
plt.legend();

In [ ]:
# scale the inputs using the standard scaler
ss_inputs = StandardScaler()

X_train_scaled = ss_inputs.fit_transform(X_train)
X_test_scaled = ss_inputs.transform(X_test)
X_train_scaled

> ### Decide, do not just run the cell
>
> The scaler is fitted on the training set and only applied to the test set.
> One line, and it is the difference between an honest score and a useless one.
>
> - What exactly leaks if you call `fit_transform` on the whole dataset before
>   splitting? Name the quantity that carries the information.
> - Would that leak make your test score look better or worse than reality?
> - We are about to save these files to disk. Does anything scaled belong in
>   them, or only the raw split?
>
> Same again: two sentences, no code.

<!-- your answer here -->

**Answer:**


## Part 4 - Save the dataset

In [ ]:
# save the training and test parts of the dataset for later usage
X_train.to_csv("../data/housing_prices/X_train.csv"); y_train.to_csv("../data/housing_prices/y_train.csv");
X_test.to_csv("../data/housing_prices/X_test.csv"); y_test.to_csv("../data/housing_prices/y_test.csv");

## Part 5 - Feature selection methods

Feature selection methods come in three families:
- *filter methods* - score each feature against the target, independently of any model,
- *wrapper methods* - train a model repeatedly and keep the features that help,
- *embedded methods* - selection comes for free from the model (Lasso, tree importances).

We run the filter methods below. Wrapper and embedded methods are in
`1x_Going_further_feature_selection`.

Note: we do not use the result - the ten features from the correlation ranking
stay. Feature selection is revisited on day 3, together with interpretability.


### Filter methods
Choose between the 10 most correlated features based on two different statistical test: Pearson correlation coefficient and mutual information measure.

In [ ]:
# import scikit-learn feature selection functions
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
# filter based feature selection function
def feature_selection(function, name, X, y):
    selection = SelectKBest(score_func=function, k='all').fit(X, y)
    feature_scores = pd.DataFrame({'features': X.columns.values, 'scores': selection.scores_})
    print(feature_scores.sort_values(by=['scores'], ascending=False))
    feature_scores.plot(x='features', kind='bar', title=name, figsize=(8, 6))
    return feature_scores

In [ ]:
f_regression_scores = feature_selection(f_regression, 'F regression - Pearson', X_train, y_train)

In [ ]:
# apply feature selection based on Mutual information measure
mutual_info_scores = feature_selection(mutual_info_regression, 'Mutual info', X_train, y_train)

In [ ]:
# compare obtained results
feature_scores = pd.DataFrame(index=relevant_inp[1:],columns=[['Pearson correlation', 'Mutual information']])
feature_scores['Pearson correlation'] = f_regression_scores.sort_values(by=['scores'], ascending=False).index
feature_scores['Mutual information'] = mutual_info_scores.sort_values(by=['scores'], ascending=False).index
feature_scores

> # HANDS-ON: Filter methods
> 
> Filter methods rank features against the target. Run the same ranking on
> different parts of the dataset and compare:
>
> ```python
> def top6(sub):
>     c = sub.select_dtypes('number').corrwith(sub['SalePrice']).abs()
>     return list(c.drop('SalePrice').sort_values(ascending=False).head(6).index)
>
> print('all      ', top6(df))
> print('pre-1980 ', top6(df[df.YearBuilt < 1980]))
> print('cheaper  ', top6(df[df.SalePrice <= df.SalePrice.median()]))
> print('expensive', top6(df[df.SalePrice > df.SalePrice.median()]))
> ```
>
> What you get:
>
> ```
> all       OverallQual, GrLivArea, GarageCars, GarageArea, TotalBsmtSF, 1stFlrSF
> pre-1980  GrLivArea, OverallQual, 1stFlrSF, Fireplaces, TotRmsAbvGrd, TotalBsmtSF
> cheaper   OverallQual, GarageCars, GarageArea, TotalBsmtSF, YearBuilt, 1stFlrSF
> expensive OverallQual, GrLivArea, TotalBsmtSF, GarageArea, GarageCars, 1stFlrSF
> ```
>
> - `GrLivArea` is second overall and missing entirely from the cheaper half. What does that say about area as a predictor?
> - `Fireplaces` appears only for pre-1980 houses. Is that a property of fireplaces or of the subset?
> - We selected ten features from the ranking on the full dataset. What did that choice assume about the houses the model will be used on?
>
> Note: a ranking is computed on a sample, not on the world. Change the sample
> and it changes the ranking.

### Your answer

<!-- write two or three sentences here -->

**Answer:**


# The end :)



_University of Zagreb Faculty of Electrical Engineering and Computing_  
_Laboratory for Renewable Energy Systems_  

_Course: AI bootcamp - Foundations of AI_  
_Notebook: 1a_EDA_data_processing_  

_References:_
- _Kaggle House Prices competition: https://www.kaggle.com/c/house-prices-advanced-regression-techniques_
- _Ames Housing Dataset (hosted on Kaggle): https://www.kaggle.com/datasets/prevek18/ames-housing-dataset_
- _FISHER, R.A. (1936), THE USE OF MULTIPLE MEASUREMENTS IN TAXONOMIC PROBLEMS. Annals of Eugenics, 7: 179-188. https://doi.org/10.1111/j.1469-1809.1936.tb02137.x_
- _Numerical Python - NumPy: https://numpy.org/_
- _Scientific Python - SciPy: https://scipy.org/_
- _Matplotlib: https://matplotlib.org/_
- _Scikit-learn Feature selection: https://scikit-learn.org/1.5/modules/feature_selection.html_
- _Scikit-learn Preprocessing data: https://scikit-learn.org/1.5/modules/preprocessing.html_
- _Scikit-learn Model selection and evaluation: https://scikit-learn.org/1.5/model_selection.html_

_Website: [www.lares.fer.hr](https://www.lares.fer.hr/)_  
_Contact: [Hrvoje Novak](mailto:hrvoje.novak@fer.hr)_